# 건설현장 산업재해 발생 영향 요인 분석

**RQ**: 건설현장의 내부 안전 관리와 실질적 안전 행동이 산업재해 발생에 미치는 영향 — 외부 기관의 조절효과를 중심으로

**분석 구조 (KEY PAPER 기반)**
- Phase 1: EDA & 기술통계
- Phase 2: Logistic Regression — 계층적 회귀 (statsmodels)
- Phase 3: ML 모델 비교 (imblearn Pipeline + SMOTENC + 5-Fold CV)
- Phase 4: SHAP 분석 (최적 모델)

## 0. 환경 설정

In [ ]:
!pip install -q statsmodels koreanize-matplotlib imbalanced-learn xgboost lightgbm shap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression as LR_sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
import shap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

In [ ]:
# 데이터 로드
df = pd.read_csv('/content/전처리_최종_v3.csv')

# 변수 그룹 정의 (RQ 기반)
IND_A  = ['안전조직수준', '위원회수준', '인증보유']
IND_B  = ['위험성평가수준', '교육훈련도움', '정리정돈상태', '작업중지권', '작업반장기여']
MOD    = ['전문지도', '고용노동부감독', '안전보건공단지원']
CTRL   = ['공사규모', '발주처', '기성공정률', '공사종류', '외국인비율']
TARGET = '사고발생'

IND_ALL      = IND_A + IND_B
ALL_FEATURES = IND_ALL + MOD + CTRL

# SMOTENC용: 외국인비율(인덱스 15)만 연속형, 나머지 15개 범주형
CAT_IDX = [i for i, col in enumerate(ALL_FEATURES) if col != '외국인비율']

X = df[ALL_FEATURES]
y = df[TARGET]

print(f"데이터: {df.shape[0]}개 사업장, {df.shape[1]}개 변수")
print(f"범주형 인덱스 (SMOTENC): {CAT_IDX}")
print(f"연속형: 외국인비율 (인덱스 {ALL_FEATURES.index('외국인비율')})")

---
## Phase 1. EDA & 기술통계

In [ ]:
# 기초 통계량
display(df.describe().round(2))

In [ ]:
# 종속변수 분포
fig, ax = plt.subplots(figsize=(5, 4))
counts = df[TARGET].value_counts().sort_index()
bars = ax.bar([0, 1], counts.values, color=['#4C72B0', '#DD8452'], width=0.5)
for bar, count in zip(bars, counts.values):
    pct = count / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{count} ({pct:.1f}%)', ha='center', fontsize=11)
ax.set_xticks([0, 1])
ax.set_xticklabels(['미발생 (0)', '발생 (1)'])
ax.set_ylabel('사업장 수')
ax.set_title('종속변수(사고발생) 분포')
plt.tight_layout()
plt.show()

In [ ]:
# 상관관계 히트맵
fig, ax = plt.subplots(figsize=(12, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, square=True, ax=ax)
ax.set_title('전체 변수 상관관계 히트맵')
plt.tight_layout()
plt.show()

In [ ]:
# VIF 다중공선성 검증
X_vif = sm.add_constant(df[ALL_FEATURES])
vif = pd.DataFrame({
    '변수명': ALL_FEATURES,
    'VIF': [variance_inflation_factor(X_vif.values, i+1) for i in range(len(ALL_FEATURES))]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

print("다중공선성(VIF) 검증 결과:")
display(vif)

---
## Phase 2. Logistic Regression — 계층적 회귀

변수를 단계적으로 투입하여 각 그룹의 설명력 기여를 확인한다.

| 모형 | 투입 변수 | 목적 |
|:---|:---|:---|
| Model 1 | 통제변수 | 기저 설명력 |
| Model 2 | + 독립A (내부 관리) | 내부 관리의 추가 설명력 |
| Model 3 | + 독립B (안전 행동) | 안전 행동의 추가 설명력 |
| Model 4 | + 조절변수 | 주효과 모형 |
| Model 5 | + 상호작용항 | 조절효과 검증 |

In [ ]:
def fit_logit(y, X_df, name):
    X_c = sm.add_constant(X_df)
    model = sm.Logit(y, X_c).fit(maxiter=1000, disp=0)

    res = model.summary2().tables[1].copy()
    res['OR'] = np.exp(res['Coef.'])
    res['OR_Lower'] = np.exp(res['[0.025'])
    res['OR_Upper'] = np.exp(res['0.975]'])
    res['Sig'] = res['P>|z|'].apply(
        lambda x: '***' if x < 0.001 else ('**' if x < 0.01 else ('*' if x < 0.05 else ''))
    )
    out = res[['Coef.', 'Std.Err.', 'z', 'P>|z|', 'Sig', 'OR', 'OR_Lower', 'OR_Upper']].round(4)

    print(f"\n[{name}]")
    print(f"  Pseudo R2={model.prsquared:.4f}  AIC={model.aic:.1f}  BIC={model.bic:.1f}  "
          f"Log-L={model.llf:.1f}  LLR p={model.llr_pvalue:.2e}")
    display(out)
    return model, out

In [ ]:
m1, s1 = fit_logit(y, df[CTRL], "Model 1: 통제변수")

In [ ]:
m2, s2 = fit_logit(y, df[CTRL + IND_A], "Model 2: 통제 + 독립A")

In [ ]:
m3, s3 = fit_logit(y, df[CTRL + IND_A + IND_B], "Model 3: 통제 + 독립A + 독립B")

In [ ]:
m4, s4 = fit_logit(y, df[ALL_FEATURES], "Model 4: 주효과 모형 (전체)")

In [ ]:
# Model 5: 조절효과 모형 (독립 x 조절 상호작용항)
X_inter = df[ALL_FEATURES].copy()
inter_terms = []
for ind in IND_ALL:
    for mod in MOD:
        name = f'{ind}x{mod}'
        X_inter[name] = X_inter[ind] * X_inter[mod]
        inter_terms.append(name)

m5, s5 = fit_logit(y, X_inter, "Model 5: 조절효과 모형")

In [ ]:
# 계층적 회귀 비교표
hier = pd.DataFrame({
    'Model': ['M1: 통제', 'M2: +독립A', 'M3: +독립A+B', 'M4: +조절', 'M5: +상호작용'],
    'Pseudo_R2': [m.prsquared for m in [m1, m2, m3, m4, m5]],
    'AIC': [m.aic for m in [m1, m2, m3, m4, m5]],
    'BIC': [m.bic for m in [m1, m2, m3, m4, m5]],
    'Log_L': [m.llf for m in [m1, m2, m3, m4, m5]],
    'n_vars': [m.df_model for m in [m1, m2, m3, m4, m5]],
}).round(4)

print("\n[계층적 회귀 모형 비교]")
display(hier)

In [ ]:
# 유의미한 상호작용항 (p < 0.1)
sig_inter = s5[s5.index.isin(inter_terms) & (s5['P>|z|'] < 0.1)]
print("[유의미한 상호작용항 (p < 0.1)]")
if len(sig_inter) > 0:
    display(sig_inter[['Coef.', 'P>|z|', 'Sig', 'OR']])
else:
    print("  없음")

---
## Phase 3. ML 모델 비교 (SMOTENC + 5-Fold CV)

**SMOTENC 사용 근거**: 16개 변수 중 15개가 정수형(이진/순서형/리커트/범주형), 연속형은 외국인비율 1개뿐. 일반 SMOTE는 이진변수에서 0.4 같은 비현실적 값을 생성하므로 SMOTENC를 사용한다.

**데이터 누수 방지**: `imblearn.pipeline.Pipeline`으로 각 CV fold 내부에서만 SMOTENC 적용.

In [ ]:
# Train/Test 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Train 사고발생 비율: {y_train.mean():.3f}")
print(f"Test  사고발생 비율: {y_test.mean():.3f}")

In [ ]:
# 모델 정의
smotenc = SMOTENC(categorical_features=CAT_IDX, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_config = {
    'Logistic Regression': {
        'model': LR_sklearn(max_iter=1000, random_state=42),
        'params': {
            'model__C': [0.01, 0.1, 1, 10],
            'model__class_weight': ['balanced', None]
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'model__n_estimators': [100, 200],
            'model__max_depth': [5, 10, None],
            'model__class_weight': ['balanced', None]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False),
        'params': {
            'model__n_estimators': [100, 200],
            'model__max_depth': [3, 5, 7],
            'model__learning_rate': [0.01, 0.1],
            'model__scale_pos_weight': [1, 2.5]
        }
    },
    'LightGBM': {
        'model': LGBMClassifier(random_state=42, verbose=-1),
        'params': {
            'model__n_estimators': [100, 200],
            'model__max_depth': [3, 5, 7],
            'model__learning_rate': [0.01, 0.1],
            'model__class_weight': ['balanced', None]
        }
    }
}

In [ ]:
# 학습 및 평가
results = []
best_models = {}

for name, cfg in models_config.items():
    print(f"--- {name} ---")

    pipe = ImbPipeline([('smote', smotenc), ('model', cfg['model'])])
    grid = GridSearchCV(pipe, cfg['params'], cv=skf, scoring='f1', n_jobs=-1, refit=True)
    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_
    yp = grid.best_estimator_.predict(X_test)
    yprob = grid.best_estimator_.predict_proba(X_test)[:, 1]

    r = {
        'Model': name,
        'Best_Params': str(grid.best_params_),
        'CV_F1': round(grid.best_score_, 4),
        'Accuracy': round(accuracy_score(y_test, yp), 4),
        'Precision': round(precision_score(y_test, yp, zero_division=0), 4),
        'Recall': round(recall_score(y_test, yp), 4),
        'F1': round(f1_score(y_test, yp), 4),
        'ROC_AUC': round(roc_auc_score(y_test, yprob), 4),
    }
    results.append(r)
    print(f"  CV F1={r['CV_F1']}  Test F1={r['F1']}  AUC={r['ROC_AUC']}")
    print(f"  Params: {grid.best_params_}\n")

results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
print("\n[ML 모델 비교 (Test Set)]")
display(results_df)

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (name, pipe) in zip(axes, best_models.items()):
    yp = pipe.predict(X_test)
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm, display_labels=['미발생', '발생']).plot(ax=ax, cmap='Blues')
    ax.set_title(name, fontsize=11)
plt.suptitle('Confusion Matrix (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve 비교
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
for (name, pipe), c in zip(best_models.items(), colors):
    yprob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, yprob)
    auc_val = roc_auc_score(y_test, yprob)
    ax.plot(fpr, tpr, color=c, lw=2, label=f'{name} (AUC={auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 최적 모델 Classification Report
best_name = results_df.iloc[0]['Model']
best_pipe = best_models[best_name]
yp_best = best_pipe.predict(X_test)

print(f"[최적 모델: {best_name}]")
print(classification_report(y_test, yp_best, target_names=['미발생', '발생']))

---
## Phase 4. SHAP 분석 (최적 모델)

In [ ]:
# SHAP 값 산출
final_model = best_pipe.named_steps['model']

if isinstance(final_model, (RandomForestClassifier, XGBClassifier, LGBMClassifier)):
    explainer = shap.TreeExplainer(final_model)
else:
    explainer = shap.LinearExplainer(final_model, X_train)

shap_values = explainer.shap_values(X_test)

# 이진분류 클래스 1 기준
if isinstance(shap_values, list):
    shap_target = shap_values[1]
elif len(shap_values.shape) == 3:
    shap_target = shap_values[:, :, 1]
else:
    shap_target = shap_values

In [ ]:
# SHAP Summary Plot (Dot)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_target, X_test, plot_type="dot", show=False, max_display=16)
plt.title(f"SHAP Summary Plot - {best_name}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot (mean |SHAP|)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_target, X_test, plot_type="bar", show=False, max_display=16)
plt.title(f"SHAP Feature Importance - {best_name}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Feature Importance 수치
shap_imp = pd.DataFrame({
    '변수명': ALL_FEATURES,
    'mean_abs_SHAP': np.abs(shap_target).mean(axis=0)
}).sort_values('mean_abs_SHAP', ascending=False).reset_index(drop=True)
shap_imp.index = range(1, len(shap_imp) + 1)
shap_imp.index.name = '순위'

print("[SHAP Feature Importance]")
display(shap_imp)

In [ ]:
# SHAP Dependence Plots - 조절효과 시각화
pairs = [
    ('정리정돈상태', '고용노동부감독'),
    ('정리정돈상태', '전문지도'),
    ('안전조직수준', '안전보건공단지원'),
    ('인증보유', '고용노동부감독'),
    ('위험성평가수준', '전문지도'),
    ('교육훈련도움', '안전보건공단지원'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, (feat, interact) in enumerate(pairs):
    ax = axes[idx // 3, idx % 3]
    plt.sca(ax)
    shap.dependence_plot(feat, shap_target, X_test, interaction_index=interact, ax=ax, show=False)
    ax.set_title(f'{feat} x {interact}', fontsize=11)
plt.suptitle(f'SHAP Dependence - 조절효과 ({best_name})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 결과 요약

In [ ]:
print("=" * 60)
print("1. 계층적 로지스틱 회귀")
print("=" * 60)
print(f"   Pseudo R2: M1={m1.prsquared:.4f} -> M4={m4.prsquared:.4f}")
print()

sig_m4 = s4[(s4['P>|z|'] < 0.05) & (s4.index != 'const')]
print("   Model 4 유의미한 변수 (p<0.05):")
for v in sig_m4.index:
    r = sig_m4.loc[v]
    d = '+' if r['OR'] > 1 else '-'
    print(f"     {v}: OR={r['OR']:.3f}({d}), p={r['P>|z|']:.4f} {r['Sig']}")

print()
print("=" * 60)
print("2. ML 모델 비교 (SMOTENC)")
print("=" * 60)
for _, r in results_df.iterrows():
    print(f"   {r['Model']}: F1={r['F1']}, AUC={r['ROC_AUC']}")
print(f"   -> 최적 모델: {best_name}")

print()
print("=" * 60)
print("3. SHAP Top-5")
print("=" * 60)
for i, (_, r) in enumerate(shap_imp.head(5).iterrows(), 1):
    print(f"   {i}. {r['변수명']} ({r['mean_abs_SHAP']:.4f})")